"""This script combines the information from 3 imported CSV files.
The first file will hold Opportunity information from the past 13 months.  Which will then be filtered down to display rows with only a referral partner and will also be joined on Account ID. 
The second file will hold product information and will be joined on Account ID, immediately filtering out uneccessary rows
The third file will hold payment information
After all files are joined we have a dataframe that includes all payments made within the last month, for which product, and the opportunity
Within the opportunity section, we are solely interested in referral partner name and want to drop all empty value rows
The resulting DF should hold all the necessary information we need to create the commission report.
Calculations are then taken, including CAD conversion from https://www.forbes.com/advisor/money-transfer/currency-converter/usd-cad/ (1 CAD = X USD)
Then export this new table to excel and send to necessary people for reporting.
Important!  Must rename files to the following:
Commission Report, pre-filter to prior year: Comission
Subscription Report, pre-filter to prior year: Subscription
Payment Report, pre-filter to 26th of prev month - 25th current month: Payment
When exporting all files from SalesForce THEY MUST BE CSV"""

In [1]:
#import Pandas, Numpy
import pandas as pd
import numpy as np
import re

#upload Payment report 
pay_df = pd.read_csv("C:/Users/LadeA/Downloads/Payment.csv")

#upload the Subscription report
sub_df = pd.read_csv("C:/Users/LadeA/Downloads/Subscription.csv")

#define a function to extract product name from embedded code because SF cant export consistently
def extract_text(text):
    match = re.search(r'>(.*?)</a>', text)
    if match:
        return match.group(1)
    else:
        return None

#Apply function as a filter and extract into new column
sub_df['Product'] = sub_df['Product Name'].apply(extract_text)

#upload the Commission report
com_df = pd.read_csv("C:/Users/LadeA/Downloads/Commission.csv")

#filter com_df to drop all rows where Referral Partner is not filled
com_df = com_df.dropna(subset='Referral Partner')
print(sub_df.head())


                        Account Name Subscription Name  \
0      Battaglia Mechanical Services       A-S00057000   
1  Sunstate Companies - Las Vegas NV       A-S00140786   
2             Superior Connects, LLC       A-S00221782   
3                   PMY Construction       A-S00295323   
4         Crossroads Building Supply       A-S00278614   

                       Subscription Charge Name       Account ID  \
0                                           CSP  0010e00001IqJ2V   
1                         PlanSwift Maintenance  0010e00001LXKJN   
2                       Insight Retail - Hotels  0010e00001KLy8N   
3                PlanSwift Professional License  0010e00001MltgR   
4  Project Intelligence Geography Missouri Plus  0010e00001LXP0X   

                                        Product Name Extended Amount Currency  \
0  <a href="a1D32000005Uarc" target="_blank">Ad S...                      CAD   
1  <a href="a1D0e000006MUiV" target="_blank">Plan...                      USD   

In [2]:
#Join all DF's.  
merged_df = com_df.merge(sub_df, on='Account Name').merge(pay_df, on="Account Name")

#drop unecessary columns 
column_list = ['PlanSwift Customer ID', 'PlanSwift Referral Date', 'Amount_x', 'Amount_y', 'Account ID_x', 'Stage', 'Created Date', 'Close Date', 'Product Name', 'Account ID_y', 'Subscription Charge Name']
merged_df.drop(labels = column_list, axis = 1, inplace=True)

#Keep only rows containing 'PlanSwift Professional' and 'PlanSwift Plugins' in the Product column
merged_df = merged_df.loc[(merged_df['Product']=='PlanSwift Professional') | (merged_df['Product']=='PlanSwift Plugins')]  

#Convert any CAD to USD
ex = 0.723526177
merged_df['Extended Amount'] = np.where((merged_df["Extended Amount Currency"]=='CAD'),merged_df['Extended Amount']*ex, merged_df['Extended Amount'])

#group 'Extended Amount' by 'Account Name' and get the sum
merged_df_group = merged_df.groupby('Account Name')['Extended Amount'].sum().round(2)
print(merged_df)

                      Account Name         Referral Partner Amount Currency_x  \
0              Tinker Masonry Inc.  Joseph Michael Ferrante               USD   
1              Tinker Masonry Inc.  Joseph Michael Ferrante               USD   
2              Tinker Masonry Inc.  Joseph Michael Ferrante               USD   
3              Tinker Masonry Inc.  Joseph Michael Ferrante               USD   
6           Oates Land Development  Joseph Michael Ferrante               USD   
7           Oates Land Development  Joseph Michael Ferrante               USD   
8           Oates Land Development  Joseph Michael Ferrante               USD   
9               Element Developers  Joseph Michael Ferrante               USD   
10              Element Developers  Joseph Michael Ferrante               USD   
11              Element Developers  Joseph Michael Ferrante               USD   
14              Blackwater Fab LLC  Joseph Michael Ferrante               USD   
15              Blackwater F

Next block we build a final dataframe for reporting and check the sum of total comission amount

In [3]:
#Build final dataframe by reseting index for a clean slate
set_df = pd.DataFrame(merged_df_group)
set_df = set_df.reset_index()

#Merge the commissions report df
final_df = set_df.merge(com_df, how = 'right')

#Add payment information
final_df = final_df.merge(pay_df, how = 'left', on='Account Name')

#Add a column for 30% commissions rate (what will be paid)
final_df['Commission Due (30%)'] = (final_df['Extended Amount']*0.3).round(2)

#Drop unecessary columns
final_df.drop(labels=['PlanSwift Referral Date'], axis = 1, inplace=True)

#Setting up a Rename Column dictionary variable because the list is continuously growing and changing
col_name_change = {'Extended Amount':'Commissionable Amount USD', 
                   'Amount_x': 'Amount', 
                   'Amount_y':'Payment Amount', 
                   'Amount Currency_x':'Amount Currency', 
                   'Effective Date':'Payment Date'}

#Rename 'Extended Amount' column to 'Commisionable Amount USD' and 'Amount_x' to 'Amount' and 'Amount_y' to 'Payment Amount' and 'Amount Currency_x' to 'Amount Currency'
final_df.rename(columns = col_name_change, inplace = True)

#Re-organize columns to fit requested format
final_df = final_df.reindex(columns=['Created Date', 
                                     'Account Name', 
                                     'PlanSwift Customer ID', 
                                     'Referral Partner', 
                                     'Amount Currency', 
                                     'Amount', 
                                     'Close Date', 
                                     'Stage', 
                                     'Commissionable Amount USD', 
                                     'Commission Due (30%)', 
                                     'Payment Number', 
                                     'Payment Date', 
                                     'Payment Amount'])

#Convert PlanSwift Customer ID to object type to avoid float.  Add 'C' to the beginning and remove decimal
final_df['PlanSwift Customer ID'] = final_df['PlanSwift Customer ID'].astype('object')
final_df['PlanSwift Customer ID'] = 'C' + final_df['PlanSwift Customer ID'].astype('str')
final_df['PlanSwift Customer ID'] = final_df['PlanSwift Customer ID'].str.replace('.0', '')

#Check final payment
sum_of_comm = final_df['Commission Due (30%)'].sum()
print(final_df)


  Created Date                    Account Name PlanSwift Customer ID  \
0    6/13/2024             Tinker Masonry Inc.               C459101   
1    5/17/2024          Oates Land Development           C2413721392   
2    7/13/2024              Element Developers           C2419463713   
3    7/15/2024              Blackwater Fab LLC           C2419696862   
4    6/19/2024  American Construction Services           C2222099051   
5    7/20/2024               Jamex Masonry LLC               C466537   
6    7/22/2024           Mountain Top Concrete               C223560   
7     7/9/2024                918 Services LLC           C2419002146   
8    7/16/2024          Moorer Mechanical Inc.           C2419722728   

          Referral Partner Amount Currency  Amount Close Date       Stage  \
0  Joseph Michael Ferrante             USD  3100.0  6/27/2024  Closed Won   
1  Joseph Michael Ferrante             USD  3200.0   7/1/2024  Closed Won   
2  Joseph Michael Ferrante             USD  2098

In [4]:
#Create Excel file with one table containing the DataFrame final_df and another table grouped by referral partner.

#table grouped by referral partner
rf_grouped = final_df.groupby('Referral Partner')['Commission Due (30%)'].sum()

#add basic information to the excel sheet
title = "Referral Partner Commission Report"
period = "Period"
per_str = "June 26 - July 25"
cad_usd_rate = "CAD/USD RATE"
cad_usd_rate_str = ex
commish = "Comission Rate"
commish_str = "0.30"

#define Excel function to loop over DF's and add to sheet
def multiple_dfs(df_list, sheet, file_name, spaces):
    writer = pd.ExcelWriter(file_name, engine = 'xlsxwriter')
    row = 5
    for dataframe in df_list:
        dataframe.to_excel(writer, sheet_name=sheet, startrow=row, startcol=0, header=False)
        row = row + len(dataframe.index) + spaces + 1
    #Formating the excel file. Get the xlsxwriter workbook and worksheet objects
    workbook = writer.book
    worksheet = writer.sheets[sheet]

    #add cell formats, adding $ to numerical columns "Amount" and "Comission Due"
    cash = workbook.add_format({'num_format': '$#,###.##'})

    #Set Each column width starting with A(0) ending with K(10), adding the cash formatting to G(6), J(9) and K(10)
    worksheet.set_column(0,0, 22.43) #A
    worksheet.set_column(1,1, 21) #B
    worksheet.set_column(2,2, 22.43) #C
    worksheet.set_column(3,3, 20.29) #D
    worksheet.set_column(4,4, 22.29) #E
    worksheet.set_column(5,5, 15.86) #F
    worksheet.set_column(6,6, 7.43, cash) #G
    worksheet.set_column(7,7, 9.71) #H
    worksheet.set_column(8,8, 10.86) #I
    worksheet.set_column(9,9, 27.29, cash) #J
    worksheet.set_column(10,10, 21, cash) #K

    #Format Header for main table
    header_format = workbook.add_format({
        "bold":True,
        "text_wrap": True,
        "valign": "top",
        "fg_color": "#D7E4BC",
        "border": 1
    })
    
    #write column headers with defined format
    for col_num, value in enumerate(final_df.columns.values):
        worksheet.write(4, col_num + 1, value, header_format)

    #add the basic information referenced above
    worksheet.write('A1', title)
    worksheet.write('A2', period)
    worksheet.write('A3', cad_usd_rate)
    worksheet.write('A4', commish)
    worksheet.write('B2', per_str)
    worksheet.write('B3', cad_usd_rate_str)
    worksheet.write('B4', commish_str)

    #Close workbook
    writer.close()

#list of dfs for function
dfs = [final_df, rf_grouped]

#excecute function
multiple_dfs(dfs, 'Commissions', 'Referral_Partner_Commissions_Report.xlsx', 1)

print(rf_grouped)

Referral Partner
Joseph Michael Ferrante    4262.32
Name: Commission Due (30%), dtype: float64
